# ResearchMentor on Colab
Run the cells **one at a time, top to bottom**. Wait for each to finish before the next.

## Step 1: Upload the project zip
Click **Choose files** and select a zip of this project (GitHub: green **Code** button > **Download ZIP**).

In [ ]:
from google.colab import files
import zipfile, glob, os
uploaded = files.upload()
zip_name = list(uploaded)[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/app')
APP_FILE = glob.glob('/content/app/**/app.py', recursive=True)[0]
APP_DIR = os.path.dirname(APP_FILE)
print('App found in:', APP_DIR)

## Step 2: Install the latest libraries
Takes 1-3 minutes. Ignore yellow warnings. If Colab asks to restart the session, do so, then re-run Step 1 only.

In [ ]:
!pip install -q -U -r {APP_DIR}/requirements.txt
!pip uninstall -y -q litellm   # not needed; avoids a package that was pulled from PyPI after a security incident
!pip list 2>/dev/null | grep -i -E "^(crewai|streamlit|google-genai) " 

## Step 3 (optional): Test a key before starting the app
The app also has a **Test my API key** button, so you can skip this. Free keys: Gemini https://aistudio.google.com/apikey , Groq https://console.groq.com/keys , OpenRouter https://openrouter.ai/keys , Mistral https://console.mistral.ai/api-keys

In [ ]:
import sys, getpass
sys.path.insert(0, APP_DIR)
from config import PROVIDERS, test_api_key
names = list(PROVIDERS)
for i, n in enumerate(names, 1): print(i, n)
provider = names[int(input('Choose provider number: ')) - 1]
models = list(PROVIDERS[provider]['models'].values())
for i, m in enumerate(models, 1): print(i, m)
model_id = models[int(input('Choose model number: ')) - 1]
key = getpass.getpass('Paste your API key: ')
print(test_api_key(provider, model_id, key))
# (True, 'Key works...') = good.  401 = new key.  404 = pick another model.  429 = wait or switch.

## Step 4: Start the app
Open the link it prints. The page asks for a **password**: it is the IP number printed just above the link. In the app sidebar choose a provider and model, paste the key, and press **Test my API key**.

In [ ]:
import os, subprocess, time, urllib.request
os.environ['PYTHONUTF8'] = '1'
!npm install -q localtunnel
subprocess.Popen(['streamlit','run',f'{APP_DIR}/app.py','--server.port','8501','--server.headless','true'], cwd=APP_DIR)
time.sleep(8)
print('Tunnel password (IP):', urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
!npx localtunnel --port 8501